# DSE Stock Forecasting Walkthrough

This notebook explores the bundled Damascus Securities Exchange dataset and evaluates next-session UIC closing-price forecasts. It uses a chronological holdout and compares fitted models with a persistence baseline.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src" / "forecast.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.forecast import load_market_data, evaluate


## Inspect the market data

The loader sorts dates chronologically and retains the last source row if the same date appears more than once.

In [ ]:
market = load_market_data(ROOT / "data" / "raw" / "dse_market_data.csv")
print(f"Rows after date cleanup: {len(market):,}")
print(f"Date range: {market['Date'].min().date()} to {market['Date'].max().date()}")
display(market[["Date", "DWX", "UIC", "gold", "oil"]].head())


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(market["Date"], market["UIC"], color="#486581")
ax.set(title="UIC historical closing price", xlabel="Market date", ylabel="UIC close")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


## Evaluate next-session forecasts

The target is the UIC close on the next available market date. Models are trained on the first 80% of dated examples and evaluated on the final 20%. Feature values come from the current session.

In [ ]:
metrics, predictions = evaluate(market)
display(metrics.round(4))
print(f"Chronological holdout observations: {len(predictions):,}")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(predictions["target_date"], predictions["actual_UIC"], label="Actual", color="#243b53", linewidth=2)
for name in ["persistence_baseline", "Linear regression", "Random forest", "XGBoost", "MLP"]:
    if name in predictions.columns:
        ax.plot(predictions["target_date"], predictions[name], label=name, alpha=0.85)
ax.set(title="UIC next-session forecasts — chronological holdout", xlabel="Target market date", ylabel="UIC close")
ax.legend(ncol=2)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


## Reading the result

Compare each model with the persistence baseline before drawing conclusions. A single chronological holdout is a useful first check, but it is not a rolling backtest and does not establish trading value. The bundled report and screenshots document the original coursework and prototype.